# 14. Исключения (Exceptions)

## Что такое исключения и зачем они нужны?

**Исключения** — это механизм обработки ошибок в Python. Простыми словами: это способ программы сказать "что-то пошло не так" и дать возможность это исправить.

### Аналогия из жизни

Представьте, что вы готовите ужин по рецепту:
1. Открыть холодильник
2. Достать молоко
3. Налить в кастрюлю

Но что если молока нет? Вы не просто "зависаете" — вы понимаете проблему и можете:
- Сходить в магазин
- Заменить молоко водой
- Приготовить другое блюдо

**Исключения в Python работают так же**: программа не просто падает, а сообщает о проблеме и даёт возможность её обработать.

### Почему исключения лучше, чем проверка на каждом шагу?

**Без исключений (плохо):**
```python
def get_user_data(user_id):
    user = database.find(user_id)
    if user is None:
        return None
    
    profile = user.get_profile()
    if profile is None:
        return None
    
    settings = profile.get_settings()
    if settings is None:
        return None
    
    return settings
```

**С исключениями (хорошо):**
```python
def get_user_data(user_id):
    try:
        return database.find(user_id).get_profile().get_settings()
    except (UserNotFound, ProfileNotFound):
        return None
```

### Философия Python: EAFP vs LBYL

- **LBYL** (Look Before You Leap) — "Посмотри, прежде чем прыгать"
- **EAFP** (Easier to Ask Forgiveness than Permission) — "Лучше просить прощения, чем разрешения"

Python предпочитает EAFP — проще попробовать и обработать ошибку, чем проверять всё заранее.


## Почему не стоит бояться исключений?

### Исключения — это НЕ ошибки программиста

Многие новички думают: "Если у меня исключение — значит я плохо написал код". Это неправда!

**Исключения — это нормальная часть работы программы:**
- Файл не найден — это не баг, это реальность (пользователь удалил файл)
- Сеть недоступна — это не баг, это интернет
- Пользователь ввёл "abc" вместо числа — это не баг, это пользователь

### Исключения — это коммуникация

Исключения — это способ одной части программы сказать другой: "Я не могу это сделать, и вот почему".

```python
# Функция честно говорит: "Я не могу разделить на ноль"
def divide(a, b):
    if b == 0:
        raise ValueError("Деление на ноль невозможно")
    return a / b
```

### Исключения делают код надёжнее

Без исключений ошибки могут "тихо" испортить данные:

```python
# Опасно: ошибка может быть незамечена
result = some_function()
if result == -1:  # Это ошибка или валидный результат?
    ...
```

С исключениями ошибка не может быть проигнорирована случайно:
```python
# Безопасно: ошибка явная
try:
    result = some_function()
except SomeError:
    # Мы точно знаем, что произошла ошибка
    ...
```

## Как читать Stacktrace (трассировку стека)?

Stacktrace — это "карта" того, как программа дошла до ошибки. Читать его нужно **снизу вверх**.

### Анатомия Stacktrace

```
Traceback (most recent call last):        # ← Заголовок: "вот как мы сюда попали"
  File "main.py", line 15, in <module>    # ← Начало: точка входа
    process_order(order_data)
  File "orders.py", line 42, in process_order  # ← Середина: промежуточные вызовы
    validate_payment(order.payment)
  File "payments.py", line 18, in validate_payment  # ← Конец: где произошла ошибка
    amount = int(payment['amount'])
KeyError: 'amount'                         # ← Тип ошибки и сообщение
```

### Правила чтения

1. **Смотри на последнюю строку** — там тип ошибки и сообщение
2. **Смотри на предпоследний блок** — там строка кода, которая упала
3. **Поднимайся вверх**, чтобы понять контекст (откуда вызвали)

### Частые типы ошибок

| Ошибка | Что случилось | Пример |
|--------|---------------|--------|
| `KeyError` | Нет такого ключа в словаре | `d['missing_key']` |
| `IndexError` | Индекс за пределами списка | `lst[100]` при len=5 |
| `TypeError` | Неправильный тип данных | `'2' + 2` |
| `ValueError` | Правильный тип, неправильное значение | `int('abc')` |
| `AttributeError` | Нет такого атрибута/метода | `'str'.append()` |
| `FileNotFoundError` | Файл не найден | `open('missing.txt')` |
| `ZeroDivisionError` | Деление на ноль | `1 / 0` |

### Практика: читаем Stacktrace

In [5]:
# Пример: цепочка вызовов, которая приводит к ошибке
# Попробуйте прочитать stacktrace снизу вверх!

def get_user_email(user_id):
    """Получает email пользователя из базы данных"""
    users_db = {1: {'name': 'Иван', 'email': 'ivan@mail.ru'},
                2: {'name': 'Мария'}}  # У Марии нет email!
    
    user = users_db[user_id]
    return user['email']  # Здесь упадёт для user_id=2

def send_notification(user_id, message):
    """Отправляет уведомление пользователю"""
    email = get_user_email(user_id)
    print(f"Отправляем '{message}' на {email}")

def process_order(order):
    """Обрабатывает заказ"""
    send_notification(order['user_id'], f"Ваш заказ #{order['id']} принят")

# Запускаем обработку заказа
order = {'id': 123, 'user_id': 10, 'amount': 1500}
process_order(order)

KeyError: 10

## 14.1 Базовый синтаксис: try / except / else / finally

### Полная структура обработки исключений

```python
try:
    # Код, который может вызвать исключение
    risky_operation()
    
except SpecificError as e:
    # Обработка конкретного типа ошибки
    handle_specific_error(e)
    
except (ErrorType1, ErrorType2):
    # Обработка нескольких типов ошибок
    handle_multiple_errors()
    
except Exception as e:
    # Обработка всех остальных ошибок (используйте осторожно!)
    log_unexpected_error(e)
    raise  # Часто лучше перебросить дальше
    
else:
    # Выполняется, если исключений НЕ было
    # Здесь код, который должен выполниться только при успехе
    success_callback()
    
finally:
    # Выполняется ВСЕГДА (даже если было исключение или return)
    # Используется для освобождения ресурсов
    cleanup()
```

### Примеры различных исключений

In [90]:
def exception_example(x):
    print(x)
    x = int(x)
    print(x)

exception_example(1)
exception_example('1')
exception_example('string')

1
1
1
1
string


ValueError: invalid literal for int() with base 10: 'string'

In [92]:
x = 1
x.isdigit()

AttributeError: 'int' object has no attribute 'isdigit'

In [6]:
data = {'x': 1, 'y': 2}
data['x']
data['z']

KeyError: 'z'

## 14.2 Обработка исключений на практике

### Важно: порядок except-блоков имеет значение!

Исключения проверяются **сверху вниз**. Более специфичные исключения должны идти **перед** общими.

In [1]:
data = {'x': 1, 'y': 2}

try:
    value = data['z']
    print(1)
except (KeyError, TypeError):
    print('Oops, there is no z key in this dictionary, setting default')
    value = 2
else:
    print('everything works properly, z is in place')


# bad
x = 1
x += 1
try:
    value = data['z']
    # write to file
    print(1)
except Exception:
    print('Oops, there is no z key in this dictionary, setting default')
    value = 2
else:
    print('everything works properly, z is in place')


if 'z' not in data:
    value = 2

value = data.get('z', 2)
print('value', value)

Oops, there is no z key in this dictionary, setting default
Oops, there is no z key in this dictionary, setting default
value 2


## 14.3 Пользовательские исключения

Иерархия встроенных исключений: https://docs.python.org/3/library/exceptions.html#exception-hierarchy

### Когда создавать свои исключения?

- Когда нужно различать разные типы ошибок в вашем приложении
- Когда хотите передать дополнительную информацию об ошибке
- Когда стандартные исключения слишком общие для вашего случая

In [16]:
class ShortInputException(ValueError): 
    '''Пользовательский класс исключения.''' 
    def __init__(self, length, atleast):
        super().__init__(self)
        self.length = length
        self.atleast = atleast

try:
    text = input('Enter anything --> ') 
    x = int(text)
    
    if len(text) < 3:
        raise ShortInputException(len(text), 3)
# except ShortInputException as ex:
#     print(f'ShortInputException: ops value is too short, {ex.length} of {ex.atleast}') 
except ValueError as e:
    print('Input is not correct, should be int > 3 symbols')
else:
    print('Everything is fine.')

Enter anything -->  12


Input is not correct, should be int > 3 symbols


In [21]:
isinstance(ShortInputException(1, 2), ValueError)

True

#### Пример работы с файлами

In [17]:
try:
    f = open('zen.txt')
    while True: 
        line = f.readline() 

        if len(line) == 0:
            break
        print(line, end='')
finally: 
    f.close()
    print('\n\nFile is closed now')

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!

File is closed now


In [18]:
with open('zen.txt') as f: 
    for line in f:
        print(line, end='')
print('\n\nFile is already closed')

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!

File is already closed


#### Пример использования в тестах

In [38]:
import re

class InvalidPhoneFormatException(ValueError):
    pass

    
def parse_phone(phone: str , is_plus_required=True) -> str:
    """Возвращает номер телефона в формате 7XXXXXXXXXX или +7XXXXXXXXXX"""
    if is_plus_required and not phone.startswith('+'):
        raise InvalidPhoneFormatException(f'Invalid phone format: {phone}')
    

    pattern = r'[^\+\d]' if is_plus_required else r'[^\d]'
    return re.sub(pattern, '', phone)


In [25]:
parse_phone('+7 (904) 9898112')

'+79049898112'

In [27]:
import ipytest
ipytest.autoconfig()

In [37]:
%%ipytest -qq
import pytest

def test_parse_phone():
    # Act & Assert
    assert parse_phone('+7(910)510-07-00') == '+79105100700'


def test_parse_phone__no_plus__raise_exception():
    # Act
    with pytest.raises(InvalidPhoneFormatException) as ex:
        parse_phone('7(910)510-07-00')

    # Assert
    assert ex.value.args[0] == 'Invalid phone format: 7(910)510-07-00'

..                                                                                           [100%]


## 14.6 Best Practices: как правильно работать с исключениями

### 1. Ловите конкретные исключения, а не `Exception`

**Плохо:**
```python
try:
    process_data(data)
except Exception:
    print("Что-то пошло не так")
```

**Хорошо:**
```python
try:
    process_data(data)
except ValueError as e:
    print(f"Неверный формат данных: {e}")
except ConnectionError as e:
    print(f"Ошибка соединения: {e}")
```

### 2. Не используйте исключения для управления потоком программы

**Плохо:**
```python
def find_user(users, name):
    try:
        for user in users:
            if user.name == name:
                raise StopIteration(user)
    except StopIteration as e:
        return e.args[0]
    return None
```

**Хорошо:**
```python
def find_user(users, name):
    for user in users:
        if user.name == name:
            return user
    return None
```

### 3. Используйте `else` для кода, который должен выполниться только при успехе

**Хорошо:**
```python
try:
    file = open('data.txt')
except FileNotFoundError:
    print("Файл не найден, создаём новый")
    data = {}
else:
    data = json.load(file)
    file.close()
```

### 4. Используйте `finally` или контекстные менеджеры для cleanup

**Хорошо:**
```python
with open('data.txt') as f:
    data = f.read()
# Файл автоматически закрыт, даже если было исключение
```

### 5. Добавляйте контекст при перебросе исключений (Python 3)

```python
try:
    user = get_user(user_id)
except DatabaseError as e:
    raise UserNotFoundError(f"Не удалось найти пользователя {user_id}") from e
```

### Практика: правильная обработка исключений

In [ ]:
# Реалистичный пример: загрузка конфигурации приложения
import json
from pathlib import Path

class ConfigError(Exception):
    """Базовое исключение для ошибок конфигурации"""
    pass

class ConfigNotFoundError(ConfigError):
    """Файл конфигурации не найден"""
    pass

class ConfigParseError(ConfigError):
    """Ошибка парсинга конфигурации"""
    pass

def load_config(config_path: str) -> dict:
    """
    Загружает конфигурацию из JSON файла.
    
    Raises:
        ConfigNotFoundError: если файл не найден
        ConfigParseError: если файл содержит невалидный JSON
    """
    path = Path(config_path)
    
    try:
        content = path.read_text(encoding='utf-8')
    except FileNotFoundError:
        raise ConfigNotFoundError(f"Конфигурация не найдена: {config_path}")
    except PermissionError:
        raise ConfigNotFoundError(f"Нет доступа к файлу: {config_path}")
    
    try:
        config = json.loads(content)
    except json.JSONDecodeError as e:
        raise ConfigParseError(f"Невалидный JSON в {config_path}: {e}")
    
    return config


# Использование
try:
    config = load_config('app_config.json')
    print(f"Конфигурация загружена: {config}")
except ConfigNotFoundError as e:
    print(f"Используем конфигурацию по умолчанию: {e}")
    config = {'debug': False, 'port': 8080}
except ConfigParseError as e:
    print(f"Критическая ошибка: {e}")
    raise  # Перебрасываем — это ошибка, которую нельзя исправить автоматически

## 14.7 Anti-patterns: как НЕ надо делать

### 1. "Pokemon Exception Handling" — ловим всё подряд

**Очень плохо:**
```python
try:
    do_something()
except:  # Ловит ВСЁ, включая KeyboardInterrupt и SystemExit!
    pass
```

**Плохо:**
```python
try:
    do_something()
except Exception:
    pass  # Молча игнорируем
```

**Почему это опасно?**
- Скрывает реальные ошибки
- Затрудняет отладку
- Может привести к повреждению данных

### 2. Слишком широкий try-блок

**Плохо:**
```python
try:
    user_input = input("Введите число: ")
    number = int(user_input)
    result = calculate(number)
    save_to_database(result)
    send_notification(user)
except ValueError:
    print("Неверный ввод")
```

**Хорошо:**
```python
user_input = input("Введите число: ")
try:
    number = int(user_input)
except ValueError:
    print("Введите корректное число")
    return

result = calculate(number)
save_to_database(result)
send_notification(user)
```

### 3. Использование исключений как "goto"

**Плохо:**
```python
class Found(Exception):
    pass

try:
    for row in matrix:
        for item in row:
            if item == target:
                raise Found()
except Found:
    print("Нашли!")
```

**Хорошо:**
```python
def find_in_matrix(matrix, target):
    for row in matrix:
        for item in row:
            if item == target:
                return True
    return False
```

### 4. Возврат None вместо исключения для ошибок

**Плохо:**
```python
def get_user(user_id):
    user = database.find(user_id)
    if user is None:
        return None  # Это ошибка или пользователь не найден?
    return user
```

**Хорошо:**
```python
def get_user(user_id):
    user = database.find(user_id)
    if user is None:
        raise UserNotFoundError(f"Пользователь {user_id} не найден")
    return user
```

### Демонстрация: к чему приводят anti-patterns

In [41]:
# Пример: "тихая" ошибка, которая приводит к потере данных

def bad_save_order(order_data):
    """ПЛОХОЙ пример: молча игнорируем все ошибки"""
    try:
        # Симулируем сохранение заказа
        if 'user_id' not in order_data:
            raise KeyError("user_id обязателен")
        print(f"Заказ сохранён: {order_data}")
        return True
    except Exception:
        pass  # "Потом разберёмся"
    return False

# Пользователь думает, что заказ сохранён, но это не так!
order = {'product': 'iPhone', 'amount': 1}  # Забыли user_id!
result = bad_save_order(order)
print(f"Результат: {result}")  # False, но мы не знаем почему!
print("Пользователь уходит, думая что заказ оформлен...")
print()

# ПРАВИЛЬНЫЙ вариант
def good_save_order(order_data):
    """Хороший пример: явно сообщаем об ошибках"""
    required_fields = ['user_id', 'product', 'amount']
    
    missing = [f for f in required_fields if f not in order_data]
    if missing:
        raise ValueError(f"Отсутствуют обязательные поля: {missing}")
    
    print(f"Заказ сохранён: {order_data}")
    return True

try:
    good_save_order(order)
except ValueError as e:
    print(f"Ошибка при сохранении заказа: {e}")
    print("Показываем пользователю форму с ошибкой")

Результат: False
Пользователь уходит, думая что заказ оформлен...

Ошибка при сохранении заказа: Отсутствуют обязательные поля: ['user_id']
Показываем пользователю форму с ошибкой


## 14.8 Реалистичные примеры

### Пример 1: Работа с внешним API

In [42]:
# Пример: Работа с внешним API с правильной обработкой ошибок
import urllib.request
import urllib.error
import json
from typing import Optional

class APIError(Exception):
    """Базовая ошибка API"""
    pass

class APIConnectionError(APIError):
    """Ошибка соединения с API"""
    pass

class APIResponseError(APIError):
    """Некорректный ответ от API"""
    def __init__(self, status_code: int, message: str):
        self.status_code = status_code
        super().__init__(f"HTTP {status_code}: {message}")

def fetch_weather(city: str) -> dict:
    """
    Получает погоду для города из API.
    
    Raises:
        APIConnectionError: нет соединения
        APIResponseError: ошибка от сервера
    """
    url = f"https://api.example.com/weather?city={city}"
    
    try:
        # В реальном коде здесь был бы настоящий запрос
        # Симулируем разные ситуации
        if city == "Москва":
            return {"temp": 15, "conditions": "облачно"}
        elif city == "НеsuperествующийГород":
            raise urllib.error.HTTPError(url, 404, "City not found", {}, None)
        else:
            raise urllib.error.URLError("Network is unreachable")
            
    except urllib.error.HTTPError as e:
        raise APIResponseError(e.code, e.reason)
    except urllib.error.URLError as e:
        raise APIConnectionError(f"Не удалось подключиться к API: {e.reason}")

# Использование в приложении
def show_weather(city: str) -> None:
    """Показывает погоду пользователю с graceful degradation"""
    try:
        weather = fetch_weather(city)
        print(f"Погода в {city}: {weather['temp']}°C, {weather['conditions']}")
        
    except APIConnectionError:
        print(f"Не удалось получить погоду для {city}. Проверьте интернет-соединение.")
        
    except APIResponseError as e:
        if e.status_code == 404:
            print(f"Город '{city}' не найден. Проверьте название.")
        else:
            print(f"Ошибка сервиса погоды: {e}")

# Тестируем разные сценарии
show_weather("Москва")
show_weather("НесуществующийГород")
show_weather("Лондон")  # Симулирует ошибку сети

Погода в Москва: 15°C, облачно
Не удалось получить погоду для НесуществующийГород. Проверьте интернет-соединение.
Не удалось получить погоду для Лондон. Проверьте интернет-соединение.


### Пример 2: Валидация данных пользователя

In [43]:
# Пример: Валидация формы регистрации
import re
from dataclasses import dataclass
from typing import List

class ValidationError(Exception):
    """Ошибка валидации с поддержкой множественных ошибок"""
    def __init__(self, errors: List[str]):
        self.errors = errors
        super().__init__(f"Ошибки валидации: {errors}")

@dataclass
class UserRegistration:
    email: str
    password: str
    age: int

def validate_registration(data: dict) -> UserRegistration:
    """
    Валидирует данные регистрации и возвращает объект UserRegistration.
    
    Raises:
        ValidationError: если данные невалидны
    """
    errors = []
    
    # Валидация email
    email = data.get('email', '').strip()
    if not email:
        errors.append("Email обязателен")
    elif not re.match(r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$', email):
        errors.append("Некорректный формат email")
    
    # Валидация пароля
    password = data.get('password', '')
    if len(password) < 8:
        errors.append("Пароль должен быть не менее 8 символов")
    if not any(c.isupper() for c in password):
        errors.append("Пароль должен содержать заглавную букву")
    if not any(c.isdigit() for c in password):
        errors.append("Пароль должен содержать цифру")
    
    # Валидация возраста
    try:
        age = int(data.get('age', 0))
        if age < 18:
            errors.append("Регистрация доступна только с 18 лет")
        if age > 120:
            errors.append("Некорректный возраст")
    except (ValueError, TypeError):
        errors.append("Возраст должен быть числом")
    
    if errors:
        raise ValidationError(errors)
    
    return UserRegistration(email=email, password=password, age=age)


# Тестируем с разными данными
test_cases = [
    {"email": "user@example.com", "password": "SecurePass123", "age": "25"},
    {"email": "invalid-email", "password": "weak", "age": "15"},
    {"email": "", "password": "", "age": "abc"},
]

for i, data in enumerate(test_cases, 1):
    print(f"--- Тест {i}: {data} ---")
    try:
        user = validate_registration(data)
        print(f"Успешно: {user}")
    except ValidationError as e:
        print(f"Ошибки:")
        for error in e.errors:
            print(f"  - {error}")
    print()

--- Тест 1: {'email': 'user@example.com', 'password': 'SecurePass123', 'age': '25'} ---
Успешно: UserRegistration(email='user@example.com', password='SecurePass123', age=25)

--- Тест 2: {'email': 'invalid-email', 'password': 'weak', 'age': '15'} ---
Ошибки:
  - Некорректный формат email
  - Пароль должен быть не менее 8 символов
  - Пароль должен содержать заглавную букву
  - Пароль должен содержать цифру
  - Регистрация доступна только с 18 лет

--- Тест 3: {'email': '', 'password': '', 'age': 'abc'} ---
Ошибки:
  - Email обязателен
  - Пароль должен быть не менее 8 символов
  - Пароль должен содержать заглавную букву
  - Пароль должен содержать цифру
  - Возраст должен быть числом



### Пример 3: Retry-логика для ненадёжных операций

In [44]:
# Пример: Retry-декоратор для ненадёжных операций
import random
import time
from functools import wraps
from typing import Tuple, Type

def retry(
    max_attempts: int = 3,
    exceptions: Tuple[Type[Exception], ...] = (Exception,),
    delay: float = 1.0
):
    """
    Декоратор для повторных попыток выполнения функции.
    
    Args:
        max_attempts: максимальное количество попыток
        exceptions: типы исключений, при которых делаем retry
        delay: задержка между попытками в секундах
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    if attempt < max_attempts:
                        print(f"Попытка {attempt}/{max_attempts} не удалась: {e}")
                        print(f"Повторяем через {delay} сек...")
                        time.sleep(delay)
                    
            raise last_exception
        return wrapper
    return decorator


# Симуляция ненадёжного сервиса
class TemporaryError(Exception):
    """Временная ошибка, которая может исчезнуть при повторе"""
    pass

attempt_counter = 0

@retry(max_attempts=3, exceptions=(TemporaryError,), delay=0.5)
def unreliable_service():
    """Сервис, который иногда падает"""
    global attempt_counter
    attempt_counter += 1
    
    if attempt_counter < 3:  # Первые 2 попытки — неудача
        raise TemporaryError("Сервер перегружен, попробуйте позже")
    
    return {"status": "ok", "data": "Важные данные"}


# Тестируем
attempt_counter = 0
try:
    result = unreliable_service()
    print(f"Успех! Результат: {result}")
except TemporaryError as e:
    print(f"Все попытки исчерпаны: {e}")

Попытка 1/3 не удалась: Сервер перегружен, попробуйте позже
Повторяем через 0.5 сек...
Попытка 2/3 не удалась: Сервер перегружен, попробуйте позже
Повторяем через 0.5 сек...
Успех! Результат: {'status': 'ok', 'data': 'Важные данные'}


### Пример 4: Транзакции и откат при ошибке

In [45]:
# Пример: Контекстный менеджер для транзакций
from contextlib import contextmanager
from typing import List, Dict

class InsufficientFundsError(Exception):
    """Недостаточно средств на счёте"""
    pass

class Account:
    """Простая модель банковского счёта"""
    def __init__(self, owner: str, balance: float):
        self.owner = owner
        self.balance = balance
        self._pending_operations: List[Dict] = []
    
    def withdraw(self, amount: float) -> None:
        if amount > self.balance:
            raise InsufficientFundsError(
                f"На счёте {self.owner} недостаточно средств: "
                f"баланс {self.balance}, требуется {amount}"
            )
        self._pending_operations.append({'type': 'withdraw', 'amount': amount})
        self.balance -= amount
    
    def deposit(self, amount: float) -> None:
        self._pending_operations.append({'type': 'deposit', 'amount': amount})
        self.balance += amount
    
    def commit(self) -> None:
        """Подтверждаем операции"""
        self._pending_operations.clear()
        print(f"  [Commit] Операции для {self.owner} подтверждены")
    
    def rollback(self) -> None:
        """Откатываем операции"""
        for op in reversed(self._pending_operations):
            if op['type'] == 'withdraw':
                self.balance += op['amount']
            else:
                self.balance -= op['amount']
        self._pending_operations.clear()
        print(f"  [Rollback] Операции для {self.owner} отменены")

@contextmanager
def transaction(*accounts: Account):
    """
    Контекстный менеджер для атомарных операций над несколькими счетами.
    При ошибке откатывает все изменения.
    """
    try:
        yield
        for account in accounts:
            account.commit()
    except Exception as e:
        for account in accounts:
            account.rollback()
        raise

def transfer(from_acc: Account, to_acc: Account, amount: float) -> None:
    """Переводит деньги между счетами атомарно"""
    print(f"Перевод {amount}₽ от {from_acc.owner} к {to_acc.owner}")
    
    with transaction(from_acc, to_acc):
        from_acc.withdraw(amount)
        to_acc.deposit(amount)
    
    print(f"  Успешно! {from_acc.owner}: {from_acc.balance}₽, {to_acc.owner}: {to_acc.balance}₽")


# Тестируем
alice = Account("Алиса", 1000)
bob = Account("Боб", 500)

print("=== Успешный перевод ===")
transfer(alice, bob, 300)

print("\n=== Неудачный перевод (недостаточно средств) ===")
try:
    transfer(alice, bob, 10000)  # У Алисы только 700₽
except InsufficientFundsError as e:
    print(f"  Ошибка: {e}")
    print(f"  Балансы не изменились: {alice.owner}: {alice.balance}₽, {bob.owner}: {bob.balance}₽")

=== Успешный перевод ===
Перевод 300₽ от Алиса к Боб
  [Commit] Операции для Алиса подтверждены
  [Commit] Операции для Боб подтверждены
  Успешно! Алиса: 700₽, Боб: 800₽

=== Неудачный перевод (недостаточно средств) ===
Перевод 10000₽ от Алиса к Боб
  [Rollback] Операции для Алиса отменены
  [Rollback] Операции для Боб отменены
  Ошибка: На счёте Алиса недостаточно средств: баланс 700, требуется 10000
  Балансы не изменились: Алиса: 700₽, Боб: 800₽


## 14.9 Итоги: Чек-лист по исключениям

### Когда использовать исключения?
- Ошибки, которые нельзя предсказать (сеть, файлы, внешние сервисы)
- Ошибки валидации данных
- Нарушение бизнес-логики (недостаточно средств, товар закончился)
- Когда нужно прервать цепочку вызовов

### Когда НЕ использовать исключения?
- Для обычного управления потоком (`if/else` лучше)
- Когда можно проверить условие заранее без затрат
- Для возврата "пустого" результата (используйте `None` или пустой список)

### Чек-лист хорошего кода с исключениями

1. **Ловлю конкретные исключения**, а не `Exception`
2. **try-блок минимальный** — только код, который может упасть
3. **Логирую исключения** для отладки
4. **Добавляю контекст** при перебросе (`raise ... from e`)
5. **Использую `finally` или `with`** для cleanup
6. **Документирую исключения** в docstring (`Raises:`)
7. **Создаю свои исключения** для своей бизнес-логики
8. **Никогда не использую голый `except:`**

### Полезные ссылки
- [Иерархия встроенных исключений](https://docs.python.org/3/library/exceptions.html#exception-hierarchy)
- [PEP 3134 — Exception Chaining](https://peps.python.org/pep-3134/)
- [contextlib — Утилиты для контекстных менеджеров](https://docs.python.org/3/library/contextlib.html)

## Упражнения

### Упражнение 1: Безопасное деление
Напишите функцию `safe_divide(a, b)`, которая:
- Возвращает результат деления a на b
- Выбрасывает `ValueError` с понятным сообщением при делении на ноль
- Выбрасывает `TypeError` если аргументы не числа

### Упражнение 2: Парсинг даты
Напишите функцию `parse_date(date_string)`, которая:
- Принимает строку в формате "ДД.ММ.ГГГГ"
- Возвращает кортеж (день, месяц, год)
- Создайте своё исключение `InvalidDateFormatError`
- Валидируйте, что день, месяц и год — корректные числа

### Упражнение 3: Контекстный менеджер для измерения памяти
Создайте контекстный менеджер `memory_usage`, который:
- Измеряет использование памяти до и после блока кода
- Выводит разницу в мегабайтах
- Подсказка: используйте `tracemalloc` из стандартной библиотеки

In [ ]:
# Упражнение 1: Безопасное деление
def safe_divide(a, b):
    # Ваш код здесь
    pass

# Тесты
# safe_divide(10, 2)  # → 5.0
# safe_divide(10, 0)  # → ValueError: Деление на ноль
# safe_divide("10", 2)  # → TypeError: Аргументы должны быть числами

In [ ]:
# Упражнение 2: Парсинг даты
class InvalidDateFormatError(Exception):
    pass

def parse_date(date_string: str):
    # Ваш код здесь
    pass

# Тесты
# parse_date("25.12.2024")  # → (25, 12, 2024)
# parse_date("2024-12-25")  # → InvalidDateFormatError
# parse_date("ab.cd.efgh")  # → InvalidDateFormatError

In [ ]:
# Упражнение 3: Контекстный менеджер для измерения памяти
import tracemalloc
from contextlib import contextmanager

@contextmanager
def memory_usage():
    # Ваш код здесь
    yield

# Тест
# with memory_usage():
#     data = [i ** 2 for i in range(1_000_000)]
# # Должно вывести что-то вроде: "Использовано памяти: 38.15 MB"